# 🛡️ AI Compliance Document Auditor — AMD Hackathon

**Use case:** Upload any compliance-sensitive document (PDF / DOCX / TXT). The app retrieves
the most relevant compliance rules via RAG, runs them through Qwen (served with vLLM on ROCm),
and returns an auditable, evidence-cited validation report — Pass/Fail, confidence score, and
a list of flagged issues with the exact quoted evidence from the document.

**Architecture**
1. **Document parsing** — PDF/DOCX/TXT → plain text, chunked to fit the model's context window.
2. **RAG retrieval** — ChromaDB + `bge-large-en-v1.5` embeddings fetch the rules relevant to each chunk.
3. **Qwen on vLLM (ROCm)** — generates a strict JSON audit report per chunk (temperature = 0 for determinism).
4. **Aggregation** — chunk-level reports are merged/deduplicated into one document-level report.
5. **Gradio app** — upload a document (and optionally your own rules file), click *Run Audit*, get a
   report you can read in the browser or download as JSON/Markdown.

Run the cells in order. The last cell launches the Gradio app with a public shareable link
(`share=True`), which is the easiest way to demo this on `notebooks.amd.com`.


## 1. Environment setup (ROCm + RAG + parsers + UI)
Run once per session/kernel.

In [ ]:
# === AMD ROCm environment setup ===
!pip install --upgrade pip --quiet
!pip install vllm --extra-index-url https://wheels.vllm.ai/rocm --quiet
!pip install chromadb sentence-transformers huggingface_hub --quiet
!pip install pdfplumber python-docx pypdf --quiet
!pip install gradio pandas --quiet

print("✅ All dependencies installed.")


In [ ]:
# === Sanity check: confirm an AMD GPU is visible to PyTorch/ROCm ===
import torch

print("torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("⚠️ No GPU detected. vLLM will fail or fall back to CPU (very slow). "
          "Check your notebooks.amd.com runtime/GPU allocation before continuing.")


## 2. Configuration

In [ ]:
import os, json, re, tempfile
from pathlib import Path

# ---- Configuration ----
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"   # swap to "OpenPipe/Qwen3-14B-Instruct" for true Qwen3 weights
TENSOR_PARALLEL_SIZE = 1                    # set to the number of GPUs you want to shard the model across
MAX_MODEL_LEN = 8192
CHUNK_CHAR_LIMIT = 3000                     # characters per document chunk sent to the LLM
CHUNK_OVERLAP = 200                         # character overlap between chunks (preserves context across boundaries)
RAG_TOP_K = 3                               # how many rules to retrieve per chunk
EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"

# Uncomment and set this if MODEL_NAME points to a gated/private HF repo:
# os.environ["HF_TOKEN"] = "hf_xxx"

print(f"Config loaded — model={MODEL_NAME}, tensor_parallel_size={TENSOR_PARALLEL_SIZE}, max_model_len={MAX_MODEL_LEN}")


## 3. Document parsing utilities (PDF / DOCX / TXT)

In [ ]:
import pdfplumber
import docx as python_docx  # the package is imported as "docx"; aliased to avoid name confusion


def extract_text_from_pdf(path: str) -> str:
    text_parts = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text_parts.append(page.extract_text() or "")
    return "\n".join(text_parts).strip()


def extract_text_from_docx(path: str) -> str:
    doc = python_docx.Document(path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    # compliance docs love tables — pull those in too
    for table in doc.tables:
        for row in table.rows:
            row_text = " | ".join(cell.text.strip() for cell in row.cells if cell.text.strip())
            if row_text:
                paragraphs.append(row_text)
    return "\n".join(paragraphs).strip()


def extract_text_from_txt(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()


def extract_text(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        return extract_text_from_pdf(path)
    elif ext == ".docx":
        return extract_text_from_docx(path)
    elif ext in (".txt", ".md"):
        return extract_text_from_txt(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}. Please upload PDF, DOCX, or TXT.")


def chunk_text(text: str, chunk_size: int = None, overlap: int = None) -> list[str]:
    """Paragraph-aware chunking with a hard-split fallback for oversized paragraphs,
    plus a small overlap so rule violations spanning a chunk boundary aren't missed."""
    chunk_size = chunk_size or CHUNK_CHAR_LIMIT
    overlap = overlap if overlap is not None else CHUNK_OVERLAP

    if len(text) <= chunk_size:
        return [text] if text.strip() else []

    paragraphs = [p for p in text.split("\n") if p.strip()]
    chunks, current = [], ""

    for para in paragraphs:
        if len(current) + len(para) + 1 <= chunk_size:
            current = f"{current}\n{para}" if current else para
        else:
            if current:
                chunks.append(current)
            if len(para) > chunk_size:
                for i in range(0, len(para), chunk_size - overlap):
                    chunks.append(para[i:i + chunk_size])
                current = ""
            else:
                current = para
    if current:
        chunks.append(current)

    overlapped = []
    for i, c in enumerate(chunks):
        if i > 0 and overlap > 0:
            tail = chunks[i - 1][-overlap:]
            c = tail + "\n" + c
        overlapped.append(c)
    return overlapped

print("✅ Document parsing utilities ready (PDF / DOCX / TXT).")


## 4. Compliance rules management (RAG)
Starts with 4 built-in example rules. Upload your own CSV (a column named `rule`, `rules`, or `rule_text` — or just the first column) or TXT file (one rule per line) from the app UI to replace them at runtime.

In [ ]:
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

DEFAULT_RULES = [
    "Rule 101: All individual investments or premium payments exceeding ₹50,000 mandate a verified Permanent Account Number (PAN) on record.",
    "Rule 102: Medical insurance policies must explicitly detail a 24-month waiting period for all pre-existing conditions.",
    "Rule 103: Corporate financial disclosures must include a clear \'Risk Factors\' section outlining market volatility impacts.",
    "Rule 104: Any performance bonus disbursement above ₹10,000 must be accompanied by a tax-deduction at source (TDS) certificate.",
]


class ComplianceRuleStore:
    """Wraps a Chroma collection so compliance rules can be swapped out at runtime
    from an uploaded CSV/TXT file, with semantic retrieval (RAG) over whatever is loaded."""

    def __init__(self, embed_model_name: str = EMBED_MODEL_NAME):
        self.client = chromadb.Client()
        self.embedder = SentenceTransformer(embed_model_name)
        self.collection = None
        self.rules: list[str] = []
        self.load_rules(DEFAULT_RULES, source_label="built-in defaults")

    def _rebuild_collection(self):
        try:
            self.client.delete_collection("compliance_rules")
        except Exception:
            pass
        self.collection = self.client.create_collection(name="compliance_rules")

    def load_rules(self, rules: list[str], source_label: str = "uploaded file"):
        rules = [r.strip() for r in rules if r and r.strip()]
        if not rules:
            raise ValueError("No valid rules found to load.")
        self._rebuild_collection()
        embeddings = self.embedder.encode(rules).tolist()
        self.collection.add(
            documents=rules,
            embeddings=embeddings,
            ids=[f"rule_{i}" for i in range(len(rules))],
        )
        self.rules = rules
        print(f"✅ Loaded {len(rules)} rule(s) from {source_label}.")

    def load_rules_from_file(self, path: str):
        ext = Path(path).suffix.lower()
        if ext == ".csv":
            df = pd.read_csv(path)
            col = next(
                (c for c in df.columns if c.strip().lower() in ("rule", "rules", "rule_text")),
                df.columns[0],
            )
            rules = df[col].astype(str).tolist()
        elif ext in (".txt", ".md"):
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                rules = [line.strip() for line in f.readlines()]
        else:
            raise ValueError(f"Unsupported rules file type: {ext}. Please upload a .csv or .txt file.")
        self.load_rules(rules, source_label=Path(path).name)

    def query(self, text: str, n_results: int = RAG_TOP_K) -> list[str]:
        n_results = min(n_results, len(self.rules)) or 1
        emb = self.embedder.encode([text]).tolist()
        results = self.collection.query(query_embeddings=emb, n_results=n_results)
        return results["documents"][0]


rule_store = ComplianceRuleStore()


## 5. Deploy Qwen via vLLM on ROCm

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
)

# temperature = 0 keeps the audit deterministic and reproducible
sampling_params = SamplingParams(temperature=0.0, max_tokens=1024)
print(f"✅ {MODEL_NAME} loaded on AMD ROCm.")


## 6. Audit engine
Builds one RAG-grounded prompt per document chunk, batches them through vLLM in a single `generate()` call, parses the JSON safely, and merges/deduplicates results into one report.

In [ ]:
JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)


def build_prompt(document_chunk: str, relevant_rules: list[str]) -> str:
    rules_block = "\n".join(relevant_rules)
    return f"""<|im_start|>system
You are a highly analytical AI Audit & Compliance Validator. Your objective is to cross-reference the provided document excerpt against the retrieved compliance rules.
You must maintain absolute consistency in your validation. Produce an auditable JSON report. Do not include any markdown formatting or text outside of the JSON block.

Structure:
{{
  "validation_status": "Pass" or "Fail",
  "overall_confidence_score": 0.0 to 1.0,
  "issues_flagged": [
    {{
      "rule_violated": "Rule ID and brief description",
      "evidence_from_document": "Exact quote from the document proving the violation",
      "explanation": "Why this evidence breaches the rule"
    }}
  ]
}}
<|im_end|>
<|im_start|>user
[Rules Context]
{rules_block}

[Document Excerpt to Audit]
{document_chunk}
<|im_end|>
<|im_start|>assistant
"""


def parse_llm_json(raw_text: str) -> dict:
    raw_text = raw_text.strip()
    raw_text = re.sub(r"^```(json)?|```$", "", raw_text, flags=re.MULTILINE).strip()
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        match = JSON_BLOCK_RE.search(raw_text)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
    # last resort: surface the raw text so nothing silently disappears
    return {
        "validation_status": "Unknown",
        "overall_confidence_score": 0.0,
        "issues_flagged": [],
        "parse_error": True,
        "raw_model_output": raw_text[:500],
    }


def audit_document(full_text: str, rules: "ComplianceRuleStore" = None) -> dict:
    rules = rules or rule_store
    chunks = chunk_text(full_text)
    if not chunks:
        return {
            "validation_status": "Unknown",
            "overall_confidence_score": 0.0,
            "issues_flagged": [],
            "chunks_audited": 0,
            "note": "Document had no extractable text.",
        }

    prompts = [build_prompt(chunk, rules.query(chunk)) for chunk in chunks]
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)

    all_issues, statuses, scores = [], [], []
    for i, out in enumerate(outputs):
        parsed = parse_llm_json(out.outputs[0].text)
        statuses.append(parsed.get("validation_status", "Unknown"))
        try:
            scores.append(float(parsed.get("overall_confidence_score", 0.0)))
        except (TypeError, ValueError):
            scores.append(0.0)
        for issue in parsed.get("issues_flagged", []):
            issue["source_chunk"] = i + 1
            all_issues.append(issue)

    # dedupe issues that show up identically across overlapping chunk boundaries
    seen, deduped = set(), []
    for issue in all_issues:
        key = (issue.get("rule_violated", ""), issue.get("evidence_from_document", ""))
        if key not in seen:
            seen.add(key)
            deduped.append(issue)

    overall_status = "Fail" if (any(s == "Fail" for s in statuses) or deduped) else "Pass"
    avg_confidence = round(sum(scores) / len(scores), 3) if scores else 0.0

    return {
        "validation_status": overall_status,
        "overall_confidence_score": avg_confidence,
        "issues_flagged": deduped,
        "chunks_audited": len(chunks),
        "rules_in_force": len(rules.rules),
    }

print("✅ Audit engine ready.")


## 7. Report generation (Markdown + downloadable files)

In [ ]:
from datetime import datetime


def report_to_markdown(report: dict, doc_name: str = "uploaded document") -> str:
    status = report.get("validation_status", "Unknown")
    badge = "🟢 PASS" if status == "Pass" else ("🔴 FAIL" if status == "Fail" else "⚪ UNKNOWN")
    lines = [
        "# Compliance Audit Report",
        f"**Document:** {doc_name}",
        f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"**Status:** {badge}",
        f"**Confidence Score:** {report.get('overall_confidence_score', 0.0)}",
        f"**Chunks Audited:** {report.get('chunks_audited', 0)}  |  **Rules Applied:** {report.get('rules_in_force', 0)}",
        "",
        "## Issues Flagged",
    ]
    issues = report.get("issues_flagged", [])
    if not issues:
        lines.append("_No issues found — document is compliant with all checked rules._")
    else:
        for i, issue in enumerate(issues, 1):
            lines.extend([
                f"### {i}. {issue.get('rule_violated', 'Unspecified rule')}",
                f"- **Evidence:** \"{issue.get('evidence_from_document', 'N/A')}\"",
                f"- **Explanation:** {issue.get('explanation', 'N/A')}",
                f"- **Source chunk:** {issue.get('source_chunk', '-')}",
                "",
            ])
    return "\n".join(lines)


def save_report_files(report: dict, doc_name: str) -> tuple[str, str]:
    tmp_dir = tempfile.mkdtemp()
    json_path = os.path.join(tmp_dir, "audit_report.json")
    md_path = os.path.join(tmp_dir, "audit_report.md")
    with open(json_path, "w") as f:
        json.dump(report, f, indent=2)
    with open(md_path, "w") as f:
        f.write(report_to_markdown(report, doc_name))
    return json_path, md_path

print("✅ Report generation utilities ready.")


## 8. Gradio web app
Launches with `share=True` so you get a public link to demo — handy on `notebooks.amd.com` where the port usually isn't reachable directly.

In [ ]:
import gradio as gr


def _filepath(f):
    """Gradio's gr.File has returned either an object with .name or a plain path string
    across versions — handle both so this keeps working regardless of the installed version."""
    if f is None:
        return None
    return f.name if hasattr(f, "name") else f


def run_audit_pipeline(document_file, rules_file):
    if document_file is None:
        return "⚠️ Please upload a document to audit.", "{}", [], None, None

    doc_path = _filepath(document_file)
    rules_path = _filepath(rules_file)
    status_prefix = ""

    if rules_path is not None:
        try:
            rule_store.load_rules_from_file(rules_path)
            status_prefix = f"Loaded custom rules from {Path(rules_path).name}. "
        except Exception as e:
            return f"❌ Failed to load rules file: {e}", "{}", [], None, None
    else:
        if rule_store.rules != DEFAULT_RULES:
            rule_store.load_rules(DEFAULT_RULES, source_label="built-in defaults (no rules file uploaded)")

    try:
        text = extract_text(doc_path)
    except Exception as e:
        return f"❌ Failed to read document: {e}", "{}", [], None, None

    if not text.strip():
        return ("⚠️ No extractable text found in the document "
                "(it may be a scanned/image-only PDF).", "{}", [], None, None)

    report = audit_document(text, rule_store)
    doc_name = Path(doc_path).name
    json_path, md_path = save_report_files(report, doc_name)

    issues_table = [
        [i.get("rule_violated", ""), i.get("evidence_from_document", ""),
         i.get("explanation", ""), i.get("source_chunk", "")]
        for i in report.get("issues_flagged", [])
    ]

    status_banner = (
        f"{status_prefix}**Status: {report['validation_status']}**  |  "
        f"Confidence: {report['overall_confidence_score']}  |  "
        f"Chunks audited: {report.get('chunks_audited', 0)}"
    )
    return status_banner, json.dumps(report, indent=2), issues_table, json_path, md_path


with gr.Blocks(title="AI Compliance Document Auditor") as app:
    gr.Markdown(
        "# 🛡️ AI Compliance Document Auditor\n"
        "Powered by Qwen on AMD ROCm + vLLM. Upload a document and, optionally, your own "
        "compliance rules file, then run the audit."
    )

    with gr.Row():
        with gr.Column(scale=1):
            doc_input = gr.File(label="Document to audit (PDF / DOCX / TXT)",
                                 file_types=[".pdf", ".docx", ".txt"])
            rules_input = gr.File(label="Compliance rules file — optional (CSV / TXT). "
                                         "Leave empty to use built-in defaults.",
                                   file_types=[".csv", ".txt"])
            run_btn = gr.Button("Run Audit", variant="primary")

        with gr.Column(scale=2):
            status_box = gr.Markdown("Status will appear here after you run an audit.")
            issues_df = gr.Dataframe(
                headers=["Rule Violated", "Evidence", "Explanation", "Chunk"],
                label="Issues Flagged", wrap=True,
            )
            with gr.Accordion("Raw JSON report", open=False):
                json_box = gr.Code(label="JSON", language="json")
            with gr.Row():
                json_download = gr.File(label="Download JSON report")
                md_download = gr.File(label="Download Markdown report")

    run_btn.click(
        fn=run_audit_pipeline,
        inputs=[doc_input, rules_input],
        outputs=[status_box, json_box, issues_df, json_download, md_download],
    )

app.launch(share=True)


## 9. Troubleshooting on AMD ROCm / notebooks.amd.com

- **401 / Repository Not Found** when loading the model → the repo ID is wrong or gated.
  `Qwen/Qwen2.5-14B-Instruct` is public; for true Qwen 3 weights with chat formatting try
  `OpenPipe/Qwen3-14B-Instruct`. For gated repos, set `os.environ["HF_TOKEN"]` in the config cell.
- **GPU not detected** → re-check the GPU check cell; confirm your `notebooks.amd.com` session
  actually has a GPU allocation, and that ROCm-enabled `torch`/`vllm` wheels installed correctly.
- **Out of memory** → lower `MAX_MODEL_LEN`, lower `TENSOR_PARALLEL_SIZE` if it's mismatched
  with available GPUs, or add `gpu_memory_utilization=0.85` (or lower) to the `LLM(...)` call.
- **Gradio link not opening** → `share=True` should print a public `*.gradio.live` URL beneath
  the cell; if it's blocked, try `app.launch(server_name="0.0.0.0", server_port=7860)` and check
  whether `notebooks.amd.com` exposes that port directly.
- **Model output isn't valid JSON** → `parse_llm_json` already falls back to regex-extracting the
  first `{...}` block; if it still fails, the raw text is preserved in `raw_model_output` for debugging.
- **Re-running after editing rules** → re-running the rules cell resets to `DEFAULT_RULES`; uploading
  a rules file in the app always overrides it for that session until you upload a document with no
  rules file again (which reloads the defaults).
